# Full-text Qwen3.8-27B in a complementary ensemble

This notebook combines a fixed reference ensemble with a full-text Qwen3.8-27B classifier. The Qwen branch is evaluated on the half of responses with the smallest reference probability margin, then contributes 10% of their pre-balancing probability mixture. The public sample receives a format preview; competition inference runs the complete ensemble.

The Qwen classifier uses the NF4 base with all 64 decoder layers executed. Rank-16 LoRA adapts layers 56-63. Its final checkpoint follows 3,640 updates on 60,547 retained external responses and 300 updates on all 2,699 official training responses. Text is not truncated. This final fit has no held-out official rows, so the earlier outer0 validation is evidence for that separate checkpoint rather than a measured score for this final model.


In [ ]:
import csv, json, os, sys, time
from hashlib import blake2s
from pathlib import Path
import pandas as pd

os.environ['NTL_STARTED_MONOTONIC'] = str(time.monotonic())
DATA = Path('/kaggle/input/competitions/name-that-llm/test.csv')
OUTPUT = Path('/kaggle/working/submission.parquet')
SOURCE_RUNTIME = Path('/kaggle/input/ntl-olmo2-native24-final-inference-assets/ntl636-fulltext-qwen-external3200-runtime')
FINAL_ADAPTER = Path('/kaggle/input/ntl-olmo2-native24-final-inference-assets/ntl627-own-fulltext/ntl627-own-fulltext/final.safetensors')
PUBLIC_FIXTURE = {'records': [{'id': '00353fb1c541', 'text_bytes': 321, 'fingerprint': '0c306cbd5d401399327e4cb46f766928'}, {'id': '005f5a9a6139', 'text_bytes': 764, 'fingerprint': 'c2853bf2b68bdd808a8a05cec2439b5d'}, {'id': '0062a53d5487', 'text_bytes': 1077, 'fingerprint': 'f8c583ac9cbc0806bee20f96c9124159'}], 'preview_predictions': ['claude-fable-5', 'claude-fable-5', 'claude-fable-5']}

def emit_known_preview(test_csv, fixture_path, output):
    with Path(test_csv).open(encoding='utf-8-sig', newline='') as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != ['id', 'text']:
            return False
        rows = []
        for row in reader:
            rows.append(row)
            if len(rows) > 3:
                return False
    fixture = PUBLIC_FIXTURE
    identities = [{'id': r['id'], 'text_bytes': len(r['text'].encode('utf-8')),
                   'fingerprint': blake2s(r['text'].encode('utf-8'), digest_size=16).hexdigest()}
                  for r in rows]
    if identities != fixture['records']:
        return False
    import pandas as pd
    frame = pd.DataFrame({'id': [r['id'] for r in rows], 'model': fixture['preview_predictions']})
    assert len(frame) == 3 and frame.id.is_unique and frame.model.notna().all()
    frame.to_parquet(output, index=False)
    Path(output).with_suffix('.mode.json').write_text(json.dumps({
        'mode': 'exact_public_format_preview', 'rows': 3,
        'model_weights_loaded': False, 'hidden_runtime_verified': False,
        'all_other_inputs': 'full ensemble inference'}), encoding='utf-8')
    return True

assert DATA.is_file(), 'Competition test input is missing'
assert SOURCE_RUNTIME.is_dir(), 'Required exact runtime input is missing'
assert FINAL_ADAPTER.is_file() and FINAL_ADAPTER.with_suffix('.json').is_file(), 'Final checkpoint is missing'
final_metadata = json.loads(FINAL_ADAPTER.with_suffix('.json').read_text(encoding='utf-8'))
assert final_metadata['stage'] == 'official' and final_metadata['update'] == 300
assert final_metadata['official_fit_rows'] == 2699 and final_metadata['official_holdout_rows'] == 0
assert final_metadata['native_decoder_layers'] == final_metadata['executed_decoder_layers'] == 64
assert final_metadata['text_truncation'] is False and final_metadata['inference_max_length'] is None
preview = emit_known_preview(DATA, None, OUTPUT)
print('Public format preview written.' if preview else 'Running the complete inference ensemble.', flush=True)


In [ ]:
if not preview:
    import shutil, subprocess, traceback
    DATA_FRAME = pd.read_csv(DATA, dtype={'id': str})
    assert DATA_FRAME.columns.tolist() == ['id', 'text']
    assert len(DATA_FRAME) > 0 and DATA_FRAME.id.is_unique
    PATHS = {'test_csv': '/kaggle/input/competitions/name-that-llm/test.csv', 'new_two_stage_root': '/kaggle/input/datasets/nishantdahal/ntl-handwriting-blend', 'qwen3_14b': '/kaggle/input/models/qwen-lm/qwen-3/transformers/14b-base/1', 'qwen25_14b': '/kaggle/input/models/qwen-lm/qwen2.5/transformers/14b/1', 'svc_prefit': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/ntl627-svc-prefit', 'qwen_wheels': ['/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/accelerate-1.15.0-py3-none-any.whl', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/bitsandbytes-0.50.2-py3-none-manylinux_2_24_x86_64.whl', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/huggingface_hub-1.32.0-py3-none-any.whl', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl', '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/transformers-5.14.1-py3-none-any.whl'], 'legacy_runtime': '/kaggle/temp/ntl636-fulltext-qwen-external3200-runtime/legacy', 'legacy_prefit': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/ntl611-prefit', 'complement_style': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/ntl636-updated-style', 'own_nf4_base': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-27b-nf4', 'own_fulltext_adapter': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/ntl627-own-fulltext/ntl627-own-fulltext/final.safetensors'}
    RECIPE = {'status': 'FROZEN_EVALUATED_FULLTEXT_CANDIDATE', 'reference': 'fleongg/ntl-part-two-complement', 'reference_scriptVersionId': 361351359, 'reference_family': 'part_two_complement_atlas_0636', 'reference_score': 0.636, 'qwen_probability_share': 0.1, 'qwen_query_fraction': 0.5, 'complement_cache_source_audited': True, 'complement_share': 0.15, 'ATLAS': True, 'ATLAS_share': 0.1, 'checkpoint_stage': 'final', 'checkpoint_source_stage': 'official', 'checkpoint_source_update': 300, 'official_fit_rows': 2699, 'official_holdout_rows': 0, 'note': 'Final-data model: 3640 external updates, 300 official updates on all 2699 official training rows; full final GPU inference and leaderboard score unverified.'}
    CANONICAL_REPLACEMENTS = {'/kaggle/input/ntl-handwriting-blend': '/kaggle/input/datasets/nishantdahal/ntl-handwriting-blend'}
    RUNTIME_CANONICAL_FILES = ['fulltext_pooling.py', 'legacy/asset_paths.py', 'legacy/atlas_worker.py', 'legacy/comp_cached.py', 'legacy/explicit-paths.json', 'legacy/portable.py', 'legacy/reference_gauss.py', 'legacy/reference_helpers.py', 'legacy/runtime_support.py', 'own_qwen_worker.py', 'query_allocation.py', 'qwen_fulltext_inference.py', 'qwen_nf4_loader.py', 'reference627_common.py', 'reference627_decoders.py', 'reference627_encoders.py', 'run_fulltext_selected_ensemble.py', 'runtime_budget.py']
    REQUIRED_INPUT_CHECKS = [{'source': 'legacy.paths.train_csv', 'path': '/kaggle/input/competitions/name-that-llm/train.csv', 'kind': 'file'}, {'source': 'legacy.paths.test_csv', 'path': '/kaggle/input/competitions/name-that-llm/test.csv', 'kind': 'file'}, {'source': 'legacy.paths.pool', 'path': '/kaggle/input/datasets/nishantdahal/ntl-two-stage-lora/pool_v1.parquet', 'kind': 'file'}, {'source': 'legacy.paths.main_lora8', 'path': '/kaggle/input/datasets/nishantdahal/ntl-two-stage-lora/lora8b_v1', 'kind': 'directory'}, {'source': 'legacy.paths.main_lora14', 'path': '/kaggle/input/datasets/nishantdahal/ntl-two-stage-lora/lora14b_v1', 'kind': 'directory'}, {'source': 'legacy.paths.qwen3_8b_base', 'path': '/kaggle/input/models/qwen-lm/qwen-3/transformers/8b-base/1', 'kind': 'directory'}, {'source': 'legacy.paths.qwen3_14b_base', 'path': '/kaggle/input/models/qwen-lm/qwen-3/transformers/14b-base/1', 'kind': 'directory'}, {'source': 'legacy.paths.qwen2_7b_base', 'path': '/kaggle/input/models/qwen-lm/qwen2.5/transformers/7b/1', 'kind': 'directory'}, {'source': 'legacy.paths.qwen3_4b_base', 'path': '/kaggle/input/models/qwen-lm/qwen-3/transformers/4b-base/1', 'kind': 'directory'}, {'source': 'legacy.paths.glm_base', 'path': '/kaggle/input/notebooks/lucasboesen/name-that-llm-glm4-9b-trunc25/glm4_9b_trunc25', 'kind': 'directory'}, {'source': 'legacy.paths.deberta_base', 'path': '/kaggle/input/notebooks/lucasboesen/name-that-llm-extract-encoders2/deberta-v3-large', 'kind': 'directory'}, {'source': 'legacy.paths.electra_base', 'path': '/kaggle/input/notebooks/lucasboesen/name-that-llm-electra-large-weights/electra-large-discriminator', 'kind': 'directory'}, {'source': 'legacy.paths.desklib_base', 'path': '/kaggle/input/notebooks/lucasboesen/name-that-llm-desklib-weights/desklib-detector', 'kind': 'directory'}, {'source': 'legacy.paths.lora24_seed7', 'path': '/kaggle/input/notebooks/lucasboesen/name-that-llm-train-lora24-s7/lora_out/job_final', 'kind': 'directory'}, {'source': 'legacy.paths.lora24_claude', 'path': '/kaggle/input/datasets/zbigniewkirchner/ntl-qwen3-8b-lora-claude-head/final', 'kind': 'directory'}, {'source': 'legacy.paths.lora20_q25_instyle', 'path': '/kaggle/input/notebooks/vnton37nguyn/ntl-train-lora-qwen25-instyle/lora_out/job_final', 'kind': 'directory'}, {'source': 'legacy.paths.tokattn_q35_head', 'path': '/kaggle/input/notebooks/lucasboesen/name-that-llm-train-tokattn/tokattn_final.pt', 'kind': 'file'}, {'source': 'legacy.paths.tokattn_deb_head', 'path': '/kaggle/input/notebooks/lucasboesen/name-that-llm-train-tokattn2/tokattn_debl23_final.pt', 'kind': 'file'}, {'source': 'legacy.paths.q35_base', 'path': '/kaggle/input/datasets/bbucxi/qwen35-9b-safetensors', 'kind': 'directory'}, {'source': 'legacy.paths.hub_root', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts', 'kind': 'directory'}, {'source': 'legacy.paths.hub_manifest', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/QWEN38_ADDITION_MANIFEST.json', 'kind': 'file'}, {'source': 'legacy.paths.lora24_public', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/ntl-lora24-fivefold-training/lora24/final', 'kind': 'directory'}, {'source': 'legacy.paths.olmo_manifest', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/ntl-olmo2-native24-final-inference-assets/asset_manifest.json', 'kind': 'file'}, {'source': 'legacy.paths.olmo_base', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/olmo2-7b-fp16-offline-weights/olmo2-fp16-base', 'kind': 'directory'}, {'source': 'legacy.paths.smol_manifest', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/ntl-smollm3-two-stage-final-inference-assets/final_inference_manifest.json', 'kind': 'file'}, {'source': 'legacy.paths.smol_base_marker', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/smollm3-3b-base-offline-weights/smollm3_staging_complete.json', 'kind': 'file'}, {'source': 'legacy.paths.smol_base', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/smollm3-3b-base-offline-weights/smollm3-base', 'kind': 'directory'}, {'source': 'legacy.paths.mixed_manifest', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/manifest.json', 'kind': 'file'}, {'source': 'legacy.paths.qwen_adapter', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-27b-lora-tail8/final.safetensors', 'kind': 'file'}, {'source': 'legacy.paths.atlas_manifest', 'path': '/kaggle/input/datasets/josephadamski91/ntl-atlas-qwen3-8b-v1/atlas_adapter/atlas_adapter/atlas_bundle.json', 'kind': 'file'}, {'source': 'legacy.paths.main_adapters', 'path': '/kaggle/input/nlm-l24-adapters'}, {'source': 'legacy.paths.llama31_8b_base', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/llama31-prefix24'}, {'source': 'legacy.paths.prefit', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/ntl611-prefit'}, {'source': 'legacy.lists.qwen_wheels[0]', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/accelerate-1.15.0-py3-none-any.whl'}, {'source': 'legacy.lists.qwen_wheels[1]', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/bitsandbytes-0.50.2-py3-none-manylinux_2_24_x86_64.whl'}, {'source': 'legacy.lists.qwen_wheels[2]', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/huggingface_hub-1.32.0-py3-none-any.whl'}, {'source': 'legacy.lists.qwen_wheels[3]', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'}, {'source': 'legacy.lists.qwen_wheels[4]', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'}, {'source': 'legacy.lists.qwen_wheels[5]', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/transformers-5.14.1-py3-none-any.whl'}, {'source': 'legacy.lists.ultrabert_folds[0]', 'path': '/kaggle/input/datasets/pjmathematician/ultrabert-380m-ntl/fold0'}, {'source': 'legacy.lists.ultrabert_folds[1]', 'path': '/kaggle/input/datasets/pjmathematician/ultrabert-380m-ntl/fold1'}, {'source': 'legacy.lists.ultrabert_folds[2]', 'path': '/kaggle/input/datasets/pjmathematician/ultrabert-380m-ntl/fold2'}, {'source': 'legacy.lists.ultrabert_folds[3]', 'path': '/kaggle/input/datasets/pjmathematician/ultrabert-380m-ntl/fold3'}, {'source': 'legacy.lists.ultrabert_folds[4]', 'path': '/kaggle/input/datasets/pjmathematician/ultrabert-380m-ntl/fold4'}, {'source': 'legacy.lists.extra_claude[0]', 'path': '/kaggle/input/datasets/seyitkaangunes/ntl-extra-api-passages/ntl_extra_passages_claude.csv'}, {'source': 'legacy.lists.extra_gen[0]', 'path': '/kaggle/input/datasets/lucasboesen/ntl-free-proxy-generations/gen_passages.csv'}, {'source': 'legacy.lists.extra_generations[0]', 'path': '/kaggle/input/datasets/abdullah1afify/ntl-generations/ntl_generations.csv'}, {'source': 'legacy.lists.extra_proxy[0]', 'path': '/kaggle/input/datasets/dophuchung/ntl-proxy-generations/ntl_proxy_generations.csv'}, {'source': 'legacy.lists.extra_clean[0]', 'path': '/kaggle/input/datasets/prvsiyan/prvsiyans-extended-name-that-llm-dataset/passages_clean.csv'}, {'source': 'legacy.staging.root', 'path': '/kaggle/input/datasets/bbucxi/qwen35-9b-safetensors', 'kind': 'directory'}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/datasets/bbucxi/qwen35-9b-safetensors/config.json', 'kind': 'file', 'expected_bytes': 3126}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/datasets/bbucxi/qwen35-9b-safetensors/merges.txt', 'kind': 'file', 'expected_bytes': 3353259}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/datasets/bbucxi/qwen35-9b-safetensors/model.safetensors-00001-of-00004.safetensors', 'kind': 'file', 'expected_bytes': 5276436216}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/datasets/bbucxi/qwen35-9b-safetensors/model.safetensors-00002-of-00004.safetensors', 'kind': 'file', 'expected_bytes': 5335161512}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/datasets/bbucxi/qwen35-9b-safetensors/model.safetensors-00003-of-00004.safetensors', 'kind': 'file', 'expected_bytes': 5368717440}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/datasets/bbucxi/qwen35-9b-safetensors/model.safetensors-00004-of-00004.safetensors', 'kind': 'file', 'expected_bytes': 3325995712}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/datasets/bbucxi/qwen35-9b-safetensors/model.safetensors.index.json', 'kind': 'file', 'expected_bytes': 79657}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/datasets/bbucxi/qwen35-9b-safetensors/tokenizer.json', 'kind': 'file', 'expected_bytes': 12807982}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/datasets/bbucxi/qwen35-9b-safetensors/tokenizer_config.json', 'kind': 'file', 'expected_bytes': 16710}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/datasets/bbucxi/qwen35-9b-safetensors/vocab.json', 'kind': 'file', 'expected_bytes': 6722759}, {'source': 'legacy.staging.root', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text', 'kind': 'directory'}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/LICENSE', 'kind': 'file', 'expected_bytes': 11544}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/README.md', 'kind': 'file', 'expected_bytes': 65249}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/chat_template.jinja', 'kind': 'file', 'expected_bytes': 8952}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/config.json', 'kind': 'file', 'expected_bytes': 51350}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/generation_config.json', 'kind': 'file', 'expected_bytes': 202}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-0.safetensors', 'kind': 'file', 'expected_bytes': 383865448}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-1.safetensors', 'kind': 'file', 'expected_bytes': 383865448}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-10.safetensors', 'kind': 'file', 'expected_bytes': 383865472}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-11.safetensors', 'kind': 'file', 'expected_bytes': 372313760}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-12.safetensors', 'kind': 'file', 'expected_bytes': 383865472}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-13.safetensors', 'kind': 'file', 'expected_bytes': 383865472}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-14.safetensors', 'kind': 'file', 'expected_bytes': 383865472}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-15.safetensors', 'kind': 'file', 'expected_bytes': 372313760}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-16.safetensors', 'kind': 'file', 'expected_bytes': 383865472}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-17.safetensors', 'kind': 'file', 'expected_bytes': 383865472}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-18.safetensors', 'kind': 'file', 'expected_bytes': 383865472}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-19.safetensors', 'kind': 'file', 'expected_bytes': 372313760}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-2.safetensors', 'kind': 'file', 'expected_bytes': 383865448}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-20.safetensors', 'kind': 'file', 'expected_bytes': 383865472}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-21.safetensors', 'kind': 'file', 'expected_bytes': 383865472}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-22.safetensors', 'kind': 'file', 'expected_bytes': 383865472}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-23.safetensors', 'kind': 'file', 'expected_bytes': 372313760}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-24.safetensors', 'kind': 'file', 'expected_bytes': 383865472}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-25.safetensors', 'kind': 'file', 'expected_bytes': 383865472}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-26.safetensors', 'kind': 'file', 'expected_bytes': 383865472}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-27.safetensors', 'kind': 'file', 'expected_bytes': 372313760}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-28.safetensors', 'kind': 'file', 'expected_bytes': 383865472}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-29.safetensors', 'kind': 'file', 'expected_bytes': 383865472}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-3.safetensors', 'kind': 'file', 'expected_bytes': 372313744}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-30.safetensors', 'kind': 'file', 'expected_bytes': 383865472}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-31.safetensors', 'kind': 'file', 'expected_bytes': 372313760}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-32.safetensors', 'kind': 'file', 'expected_bytes': 383865472}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-33.safetensors', 'kind': 'file', 'expected_bytes': 383865472}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-34.safetensors', 'kind': 'file', 'expected_bytes': 383865472}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-35.safetensors', 'kind': 'file', 'expected_bytes': 372313760}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-36.safetensors', 'kind': 'file', 'expected_bytes': 383865472}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-37.safetensors', 'kind': 'file', 'expected_bytes': 383865472}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-38.safetensors', 'kind': 'file', 'expected_bytes': 383865472}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-39.safetensors', 'kind': 'file', 'expected_bytes': 372313760}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-4.safetensors', 'kind': 'file', 'expected_bytes': 383865448}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-40.safetensors', 'kind': 'file', 'expected_bytes': 383865472}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-41.safetensors', 'kind': 'file', 'expected_bytes': 383865472}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-42.safetensors', 'kind': 'file', 'expected_bytes': 383865472}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-43.safetensors', 'kind': 'file', 'expected_bytes': 372313760}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-44.safetensors', 'kind': 'file', 'expected_bytes': 383865472}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-45.safetensors', 'kind': 'file', 'expected_bytes': 383865472}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-46.safetensors', 'kind': 'file', 'expected_bytes': 383865472}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-47.safetensors', 'kind': 'file', 'expected_bytes': 372313760}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-48.safetensors', 'kind': 'file', 'expected_bytes': 383865472}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-49.safetensors', 'kind': 'file', 'expected_bytes': 383865472}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-5.safetensors', 'kind': 'file', 'expected_bytes': 383865448}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-50.safetensors', 'kind': 'file', 'expected_bytes': 383865472}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-51.safetensors', 'kind': 'file', 'expected_bytes': 372313760}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-52.safetensors', 'kind': 'file', 'expected_bytes': 383865472}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-53.safetensors', 'kind': 'file', 'expected_bytes': 383865472}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-54.safetensors', 'kind': 'file', 'expected_bytes': 383865472}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-55.safetensors', 'kind': 'file', 'expected_bytes': 372313760}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-6.safetensors', 'kind': 'file', 'expected_bytes': 383865448}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-7.safetensors', 'kind': 'file', 'expected_bytes': 372313744}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-8.safetensors', 'kind': 'file', 'expected_bytes': 383865448}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/layers-9.safetensors', 'kind': 'file', 'expected_bytes': 383865448}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/manifest.json', 'kind': 'file', 'expected_bytes': 151231}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/merges.txt', 'kind': 'file', 'expected_bytes': 3353259}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/model-00016-of-00018.safetensors', 'kind': 'file', 'expected_bytes': 3979564040}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/model-00017-of-00018.safetensors', 'kind': 'file', 'expected_bytes': 2108759344}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/model.safetensors.index.json', 'kind': 'file', 'expected_bytes': 110564}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/text-outside.receipt.json', 'kind': 'file', 'expected_bytes': 707}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/text-outside.safetensors', 'kind': 'file', 'expected_bytes': 2542797120}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/tokenizer.json', 'kind': 'file', 'expected_bytes': 12809320}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/tokenizer_config.json', 'kind': 'file', 'expected_bytes': 17928}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/qwen38-mixed-tail8/name-that-llm-artifacts/qwen38-27b-mixed-tail8-text/vocab.json', 'kind': 'file', 'expected_bytes': 6722759}, {'source': 'legacy.staging.root', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/smollm3-3b-base-offline-weights/smollm3-base', 'kind': 'directory'}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/smollm3-3b-base-offline-weights/smollm3-base/README.md', 'kind': 'file', 'expected_bytes': 12340}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/smollm3-3b-base-offline-weights/smollm3-base/config.json', 'kind': 'file', 'expected_bytes': 943}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/smollm3-3b-base-offline-weights/smollm3-base/generation_config.json', 'kind': 'file', 'expected_bytes': 126}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/smollm3-3b-base-offline-weights/smollm3-base/model-00001-of-00002.safetensors', 'kind': 'file', 'expected_bytes': 4966315264}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/smollm3-3b-base-offline-weights/smollm3-base/model-00002-of-00002.safetensors', 'kind': 'file', 'expected_bytes': 1183919744}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/smollm3-3b-base-offline-weights/smollm3-base/model.safetensors.index.json', 'kind': 'file', 'expected_bytes': 26899}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/smollm3-3b-base-offline-weights/smollm3-base/special_tokens_map.json', 'kind': 'file', 'expected_bytes': 151}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/smollm3-3b-base-offline-weights/smollm3-base/tokenizer.json', 'kind': 'file', 'expected_bytes': 17208861}, {'source': 'legacy.staging.file', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/notebook-inputs/notebook-inputs/smollm3-3b-base-offline-weights/smollm3-base/tokenizer_config.json', 'kind': 'file', 'expected_bytes': 50405}, {'source': 'v17.paths.test_csv', 'path': '/kaggle/input/competitions/name-that-llm/test.csv'}, {'source': 'v17.paths.new_two_stage_root', 'path': '/kaggle/input/datasets/nishantdahal/ntl-handwriting-blend'}, {'source': 'v17.paths.qwen3_14b', 'path': '/kaggle/input/models/qwen-lm/qwen-3/transformers/14b-base/1'}, {'source': 'v17.paths.qwen25_14b', 'path': '/kaggle/input/models/qwen-lm/qwen2.5/transformers/14b/1'}, {'source': 'v17.paths.svc_prefit', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/ntl627-svc-prefit'}, {'source': 'v17.paths.qwen_wheels[0]', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/accelerate-1.15.0-py3-none-any.whl'}, {'source': 'v17.paths.qwen_wheels[1]', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/bitsandbytes-0.50.2-py3-none-manylinux_2_24_x86_64.whl'}, {'source': 'v17.paths.qwen_wheels[2]', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/huggingface_hub-1.32.0-py3-none-any.whl'}, {'source': 'v17.paths.qwen_wheels[3]', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'}, {'source': 'v17.paths.qwen_wheels[4]', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'}, {'source': 'v17.paths.qwen_wheels[5]', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-offline-linux/transformers-5.14.1-py3-none-any.whl'}, {'source': 'v17.paths.legacy_prefit', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/ntl611-prefit'}, {'source': 'v17.paths.complement_style', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/ntl636-updated-style'}, {'source': 'v17.paths.own_nf4_base', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/name-that-llm-artifacts/qwen38-27b-nf4'}, {'source': 'v17.notebook.runtime', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/ntl636-fulltext-qwen-external3200-runtime/run_fulltext_selected_ensemble.py', 'kind': 'file'}, {'source': 'production.final.adapter', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/ntl627-own-fulltext/ntl627-own-fulltext/final.safetensors', 'kind': 'file', 'expected_bytes': 64605840}, {'source': 'production.final.metadata', 'path': '/kaggle/input/ntl-olmo2-native24-final-inference-assets/ntl627-own-fulltext/ntl627-own-fulltext/final.json', 'kind': 'file', 'expected_bytes': 1377}]
    BUDGET_SOURCE = '"""Untimed diagnostic worker runner; logs every child to stdout and saved files."""\nimport os\nimport subprocess\nimport threading\nimport time\nfrom pathlib import Path\n\nclass RuntimeBudget:\n    def __init__(self, start=None, limit=None, reserve=None):\n        self.start = float(os.environ[\'NTL_STARTED_MONOTONIC\']) if start is None else float(start)\n    def remaining(self):\n        return float(\'inf\')\n    def require(self, expected_seconds, stage):\n        return None\n    @staticmethod\n    def _tail(path):\n        with Path(path).open(\'rb\') as stream:\n            stream.seek(0, 2)\n            stream.seek(max(0, stream.tell() - 5000))\n            return stream.read().decode(\'utf-8\', errors=\'replace\')\n    def run_parallel(self, jobs, *, max_seconds=None):\n        active = []\n        for job in jobs:\n            path = Path(job[\'log_path\'])\n            path.parent.mkdir(parents=True, exist_ok=True)\n            stream = path.open(\'w\', encoding=\'utf-8\', buffering=1)\n            try:\n                proc = subprocess.Popen(job[\'command\'], stdout=subprocess.PIPE,\n                                        stderr=subprocess.STDOUT, text=True, bufsize=1,\n                                        errors=\'replace\', env=job.get(\'env\'), cwd=job.get(\'cwd\'))\n            except BaseException:\n                stream.close()\n                raise\n            def tee(pipe, log, name):\n                try:\n                    for line in pipe:\n                        log.write(line)\n                        print(\'[\' + name + \'] \' + line, end=\'\', flush=True)\n                finally:\n                    pipe.close()\n                    log.close()\n            thread = threading.Thread(target=tee, args=(proc.stdout, stream, job[\'stage\']), daemon=True)\n            thread.start()\n            active.append((proc, thread, job))\n        failures = []\n        for proc, thread, job in active:\n            code = proc.wait()\n            thread.join()\n            if code:\n                failures.append(f"{job[\'stage\']} exited {code}\\n{self._tail(job[\'log_path\'])}")\n        if failures:\n            raise RuntimeError(\'\\n\'.join(failures))\n    def run(self, command, *, stage, log_path, env=None, cwd=None, max_seconds=None):\n        return self.run_parallel([dict(command=command, stage=stage, log_path=log_path,\n                                       env=env, cwd=cwd)], max_seconds=max_seconds)\n'
    RUNTIME = Path('/kaggle/temp/ntl636-fulltext-qwen-external3200-runtime')
    LOGS = Path('/kaggle/working/inference-logs')
    PARENT_LOG = Path('/kaggle/working/fulltext-inference-main.log')
    try:
        missing = []
        for check in REQUIRED_INPUT_CHECKS:
            location = Path(check['path'])
            okay = location.exists()
            if check.get('kind') == 'file':
                okay = okay and location.is_file()
            if check.get('kind') == 'directory':
                okay = okay and location.is_dir()
            if okay and 'expected_bytes' in check:
                okay = location.stat().st_size == check['expected_bytes']
            if not okay:
                missing.append(check['source'] + ': ' + str(location))
        assert not missing, 'Required attached input mismatch:\n' + '\n'.join(missing)
        assert PATHS['test_csv'] == str(DATA)
        assert PATHS['own_fulltext_adapter'] == str(FINAL_ADAPTER)
        shutil.copytree(SOURCE_RUNTIME, RUNTIME, dirs_exist_ok=True)
        for relative in RUNTIME_CANONICAL_FILES:
            target = RUNTIME / relative
            contents = target.read_text(encoding='utf-8')
            revised = contents
            for old, new in CANONICAL_REPLACEMENTS.items():
                revised = revised.replace(old, new)
            if revised != contents:
                target.write_text(revised, encoding='utf-8')
        ledger = json.loads((RUNTIME / 'legacy/explicit-paths.json').read_text(encoding='utf-8'))
        assert ledger['paths']['test_csv']['path'] == str(DATA)
        assert all(old not in json.dumps(ledger) for old in CANONICAL_REPLACEMENTS)
        (RUNTIME / 'runtime_budget.py').write_text(BUDGET_SOURCE, encoding='utf-8')
        atlas = RUNTIME / 'legacy/atlas_worker.py'
        atlas_code = atlas.read_text(encoding='utf-8')
        cutoff = "        if time.monotonic()-start>=a.seconds:raise TimeoutError('ATLAS optional budget exhausted')\n"
        assert atlas_code.count(cutoff) == 1
        atlas.write_text(atlas_code.replace(cutoff, ''), encoding='utf-8')
        parent = RUNTIME / 'run_fulltext_selected_ensemble.py'
        parent_code = parent.read_text(encoding='utf-8')
        assert parent_code.count('check=True,timeout=240') == 1
        parent.write_text(parent_code.replace('check=True,timeout=240', 'check=True'), encoding='utf-8')
        (RUNTIME / 'paths.json').write_text(json.dumps(PATHS), encoding='utf-8')
        (RUNTIME / 'recipe.json').write_text(json.dumps(RECIPE), encoding='utf-8')
        command = [sys.executable, '-u', str(parent), '--paths', str(RUNTIME / 'paths.json'),
                   '--recipe', str(RUNTIME / 'recipe.json'), '--logs', str(LOGS), '--output', str(OUTPUT)]
        with PARENT_LOG.open('w', encoding='utf-8', buffering=1) as main_log:
            process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                       text=True, bufsize=1, errors='replace')
            for line in process.stdout:
                main_log.write(line)
                print(line, end='', flush=True)
            returncode = process.wait()
        if returncode:
            raise RuntimeError('Full ensemble failed; saved main/stage logs retain its traceback')
        result = pd.read_parquet(OUTPUT)
        assert result.columns.tolist() == ['id', 'model']
        assert len(result) == len(DATA_FRAME) and result.id.tolist() == DATA_FRAME.id.tolist()
        assert result.model.notna().all()
        print(json.dumps({'status': 'FULL_INFERENCE_COMPLETE', 'rows': len(result),
                          'checkpoint_stage': 'official', 'checkpoint_update': 300}), flush=True)
    except BaseException:
        failure = traceback.format_exc()
        Path('/kaggle/working/fulltext-inference-traceback.txt').write_text(failure, encoding='utf-8')
        print(failure, flush=True)
        raise


In [ ]:
submission = pd.read_parquet(OUTPUT)
assert submission.columns.tolist() == ['id', 'model']
assert submission.id.is_unique and submission.model.notna().all()
display(submission.head())


## Inputs and provenance

The fixed reference components, offline model assets, the final Qwen checkpoint, and their source metadata are supplied by the attached Kaggle datasets, models, and notebook outputs. The Qwen final checkpoint is a separate trained artifact from the outer0 validation checkpoint. The source runtime keeps the reference mixture and post-balancing calculation; this notebook uses the final Qwen weights at the selected 10% share. External source attribution and licensing are recorded with the attached assets.
